# 02 · Preparación de datos → Silver

Bronze es una copia intocable de lo que había en el sistema operacional. **Silver es la primera
capa donde se toman decisiones**, y cada una queda argumentada aquí antes de escribirse en código.

### Qué entra y qué no

Silver contiene únicamente **limpieza determinística**: tipado, deduplicación, exclusión de
columnas y features que no aprenden nada de los datos (`balance_zero`, `age_group`).

Lo que **aprende estadísticos** —imputadores, codificadores, escaladores— no entra aquí. Va
dentro del `Pipeline` de sklearn en el notebook 04, ajustándose solo con los datos de
entrenamiento. Ese es el control contra la fuga de datos, y la razón por la que Silver no escala
ni codifica nada.

### Sobre las herramientas

El enunciado pide usar Python, PySpark y SQL. Aquí se usan los tres donde cada uno rinde:
**PySpark** para las transformaciones que producen Silver, **pandas** para el diagnóstico y el
perfilado —10.000 filas caben de sobra en memoria y la exploración es mucho más ágil— y **SQL**
para las consultas de verificación.

### Este notebook no toca Postgres

Silver se construye de Delta a Delta. Lakebase no vuelve a aparecer hasta el notebook 06, cuando
las predicciones viajan de vuelta al sistema operacional.

## Entorno

In [0]:
import pandas as pd
from pyspark.sql import functions as F

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

CATALOG       = "bank_churn"
BRONZE_TABLE  = f"{CATALOG}.bronze.bank_customers_raw"
SILVER_SCHEMA = f"{CATALOG}.silver"
SILVER_TABLE  = f"{SILVER_SCHEMA}.bank_customers_clean"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")

bronze = spark.table(BRONZE_TABLE)
print("filas en Bronze:", bronze.count())
bronze.printSchema()

filas en Bronze: 10000
root
 |-- row_number: integer (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- surname: string (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- geography: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- balance: decimal(14,2) (nullable = true)
 |-- num_of_products: integer (nullable = true)
 |-- has_cr_card: short (nullable = true)
 |-- is_active_member: short (nullable = true)
 |-- estimated_salary: decimal(14,2) (nullable = true)
 |-- exited: short (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_url: string (nullable = true)
 |-- _extracted_at: timestamp (nullable = true)



In [0]:
# Copia en pandas para diagnóstico. Las columnas DECIMAL llegan como Decimal:
# se convierten a float para poder operar con ellas.
df = bronze.toPandas()

for c in ["balance", "estimated_salary"]:
    df[c] = df[c].astype(float)

# Columnas técnicas de trazabilidad: se conservan en Bronze, se excluyen del análisis
META = ["_ingested_at", "_source_file", "_source_url", "_extracted_at"]
META = [c for c in META if c in df.columns]

print("dimensiones:", df.shape)
print("metadatos excluidos del análisis:", META)
df.head(3)

dimensiones: (10000, 18)
metadatos excluidos del analisis: ['_ingested_at', '_source_file', '_source_url', '_extracted_at']


,row_number,customer_id,surname,credit_score,geography,gender,age,tenure,balance,num_of_products,has_cr_card,is_active_member,estimated_salary,exited,_ingested_at,_source_file,_source_url,_extracted_at
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1,2026-08-02 00:59:47.485306,churn.csv,https://www.kaggle.com/datasets/mathchi/churn-...,2026-08-02 01:00:00.531356
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,2026-08-02 00:59:47.485306,churn.csv,https://www.kaggle.com/datasets/mathchi/churn-...,2026-08-02 01:00:00.531356
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,2026-08-02 00:59:47.485306,churn.csv,https://www.kaggle.com/datasets/mathchi/churn-...,2026-08-02 01:00:00.531356


## A · Auditoría de calidad — estado «antes»

La rúbrica pide comparar el perfil de calidad antes y después de la preparación. Este es el
«antes», tomado sobre Bronze sin ninguna modificación.

In [0]:
analysis_cols = [c for c in df.columns if c not in META]

profile_before = pd.DataFrame({
    "dtype":      df[analysis_cols].dtypes.astype(str),
    "n_null":     df[analysis_cols].isna().sum(),
    "pct_null":   (df[analysis_cols].isna().mean() * 100).round(2),
    "n_unique":   df[analysis_cols].nunique(),
    "cardinalidad": (df[analysis_cols].nunique() / len(df) * 100).round(2),
})
profile_before

,dtype,n_null,pct_null,n_unique,cardinalidad
row_number,int32,0,0.0,10000,100.00
customer_id,int64,0,0.0,10000,100.00
surname,object,0,0.0,2932,29.32
credit_score,int32,0,0.0,460,4.60
geography,object,0,0.0,3,0.03
gender,object,0,0.0,2,0.02
age,int32,0,0.0,70,0.70
tenure,int32,0,0.0,11,0.11
balance,float64,0,0.0,6382,63.82
num_of_products,int32,0,0.0,4,0.04


In [0]:
# Estadísticos de las numéricas: rangos, dispersión y asimetría
num_cols = ["credit_score", "age", "tenure", "balance",
            "num_of_products", "estimated_salary"]

desc = df[num_cols].describe().T
desc["skew"]     = df[num_cols].skew()
desc["kurtosis"] = df[num_cols].kurtosis()
desc.round(2)

,count,mean,std,min,25%,50%,75%,max,skew,kurtosis
credit_score,10000.0,650.53,96.65,350.00,584.00,652.00,718.00,850.00,-0.07,-0.43
age,10000.0,38.92,10.49,18.00,32.00,37.00,44.00,92.00,1.01,1.40
tenure,10000.0,5.01,2.89,0.00,3.00,5.00,7.00,10.00,0.01,-1.17
balance,10000.0,76485.89,62397.41,0.00,0.00,97198.54,127644.24,250898.09,-0.14,-1.49
num_of_products,10000.0,1.53,0.58,1.00,1.00,1.00,2.00,4.00,0.75,0.58
estimated_salary,10000.0,100090.24,57510.49,11.58,51002.11,100193.92,149388.25,199992.48,0.00,-1.18


In [0]:
# Balance de la variable objetivo
target = df["exited"].value_counts().rename({0: "permanece", 1: "abandona"})
print(target.to_string())
print(f"\ntasa de abandono : {df['exited'].mean():.2%}")
print(f"ratio de desbalance : 1 a {(1 - df['exited'].mean()) / df['exited'].mean():.1f}")

exited
permanece    7963
abandona     2037

tasa de abandono : 20.37%
ratio de desbalance : 1 a 3.9


---

## B · Decisión 1 — ¿Qué es `balance = 0`?

Es la primera decisión de fondo del notebook, y no tiene una respuesta obvia. Hay tres lecturas
posibles y cada una lleva a un tratamiento distinto:

**(a) Es un valor legítimo.** El cliente tiene la cuenta a cero. Se conserva tal cual y se crea
`balance_zero` como bandera explícita.

**(b) Es un nulo disfrazado.** El sistema de origen no tenía el dato y escribió cero por defecto.
Entonces habría que imputarlo, porque un cero real arrastraría la media hacia abajo y distorsionaría
cualquier estadístico.

**(c) Es otro tipo de producto.** Una cuenta de crédito o de valores que no lleva saldo asociado.
En ese caso `balance` estaría mezclando dos poblaciones distintas y el cero sería, en realidad,
una variable categórica encubierta.

La celda siguiente aporta la evidencia. **La tabla por número de productos es la decisiva**: si
los ceros se reparten de forma uniforme entre 1, 2, 3 y 4 productos, el patrón es aleatorio y
apunta a (b). Si se concentran en un valor concreto, no es azar sino estructura, y apunta a (c).

In [0]:
zero = df["balance"] == 0

print(f"balance = 0 : {zero.sum()} ({zero.mean():.1%})")
print(f"abandono si balance = 0 : {df.loc[zero,  'exited'].mean():.1%}")
print(f"abandono si balance > 0 : {df.loc[~zero, 'exited'].mean():.1%}")

print("\n% de saldo cero por país:")
print(df.assign(cero=zero).groupby("geography")["cero"].mean().mul(100).round(1).to_string())

print("\n% de saldo cero por número de productos:")
print(df.assign(cero=zero).groupby("num_of_products")["cero"]
        .agg(pct_cero=lambda s: s.mean() * 100, n="size").round(1).to_string())

print("\ndistribución de balance cuando es > 0:")
print(df.loc[~zero, "balance"].describe().round(0).to_string())

balance = 0 : 3617 (36.2%)
abandono si balance = 0 : 13.8%
abandono si balance > 0 : 24.1%

% de saldo cero por país:
geography
France     48.2
Germany     0.0
Spain      48.4

% de saldo cero por número de productos:
                 pct_cero     n
num_of_products                
1                    17.8  5084
2                    56.6  4590
3                    36.8   266
4                    23.3    60

distribución de balance cuando es > 0:
count      6383.0
mean     119827.0
std       30095.0
min        3769.0
25%      100182.0
50%      119840.0
75%      139512.0
max      250898.0


El primer resultado parece concluyente: 13,8% de abandono con saldo cero frente a 24,1% con saldo.
Pero **Alemania no tiene ni un solo cliente con saldo cero**, y abandona al 32,4%. Eso significa que
los 2.509 alemanes están todos en el grupo «con saldo», inflándolo.

Antes de concluir nada hay que aislar ese confusor.

In [0]:
sin_alemania = df[df["geography"] != "Germany"].copy()
z = sin_alemania["balance"] == 0

print("— solo Francia y España —")
print(f"balance = 0 : {z.sum():5d}  abandono {sin_alemania.loc[z,  'exited'].mean():.1%}")
print(f"balance > 0 : {(~z).sum():5d}  abandono {sin_alemania.loc[~z, 'exited'].mean():.1%}")

print("\n— productos, sin el sesgo alemán —")
print(sin_alemania.assign(cero=z).groupby("num_of_products")["cero"]
      .agg(pct_cero=lambda s: s.mean()*100, n="size").round(1).to_string())

print("\n— abandono cruzando país y saldo —")
print(df.assign(cero=df["balance"] == 0)
        .pivot_table(index="geography", columns="cero", values="exited",
                     aggfunc=["mean", "size"]).round(3).to_string())

— solo Francia y España —
balance = 0 :  3617  abandono 13.8%
balance > 0 :  3874  abandono 18.7%

— productos, sin el sesgo alemán —
                 pct_cero     n
num_of_products                
1                    24.2  3735
2                    73.2  3550
3                    57.6   170
4                    38.9    36

— abandono cruzando país y saldo —
            mean           size        
cero       False  True    False   True 
geography                              
France     0.182  0.139  2596.0  2418.0
Germany    0.324    NaN  2509.0     NaN
Spain      0.196  0.136  1278.0  1199.0


### El efecto existe, pero es la mitad de grande

| | aparente | aislado |
|---|---|---|
| balance = 0 | 13,8% | 13,8% |
| balance > 0 | **24,1%** | **18,7%** |
| diferencia | 10,3 pp | **4,9 pp** |

Y la tabla cruzada aporta mejor evidencia que cualquier test agregado:

```
France   sin saldo 13,9%  ·  con saldo 18,2%   →  +4,3 pp
Spain    sin saldo 13,6%  ·  con saldo 19,6%   →  +6,0 pp
```

**El efecto se replica en dos países independientes**, mismo signo y magnitud similar, cada uno
por encima de cuatro errores estándar. Una casualidad tendría que repetirse dos veces.

### Los productos: hay estructura, no inventamos el mecanismo

Sin el sesgo alemán, el saldo cero se concentra de forma muy desigual (24,2% / 73,2% / 57,6% /
38,9%). Es estructura evidente, pero **no hay forma de saber por qué**: el dataset no trae
catálogo de productos ni nombres. Se documenta como asociación observada, sin construir una
narrativa de negocio que suene bien pero no esté respaldada.

### ¿Importa el nivel de saldo, o solo el cero?

Si entre quienes tienen saldo el nivel no discrimina, entonces `balance_zero` captura toda la
información y la variable continua sobra. Y de paso, comparar el perfil de los tres países debería
revelar de dónde salen los trece puntos extra de Alemania.

In [0]:
con_saldo = df[df["balance"] > 0].copy()
con_saldo["q"] = pd.qcut(con_saldo["balance"], 5, labels=["Q1","Q2","Q3","Q4","Q5"])
print("abandono por quintil de saldo (solo saldo > 0):")
print(con_saldo.groupby("q", observed=True)["exited"]
      .agg(abandono=lambda s: round(s.mean()*100, 1), n="size").to_string())

print("\nperfil comparado por país:")
print(df.groupby("geography").agg(
    edad_media=("age", "mean"),
    pct_activos=("is_active_member", "mean"),
    productos_medio=("num_of_products", "mean"),
    saldo_medio=("balance", "mean"),
    abandono=("exited", "mean"),
).round(3).to_string())

abandono por quintil de saldo (solo saldo > 0):
    abandono     n
q                 
Q1      20.8  1277
Q2      25.2  1276
Q3      26.8  1277
Q4      24.8  1276
Q5      22.9  1277

perfil comparado por país:
           edad_media  pct_activos  productos_medio  saldo_medio  abandono
geography                                                                 
France         38.512        0.517            1.531    62092.637     0.162
Germany        39.772        0.497            1.520   119730.116     0.324
Spain          38.891        0.530            1.539    61818.148     0.167


### Decisión 1 · `balance = 0` es un estado legítimo y distinto

Evidencia acumulada:

- **Hueco en la distribución**: no existe ningún saldo entre 0 y 3.769. Un nulo codificado como
  cero no produce un salto así; una masa puntual en 0 más una campaña en 120.000 sí.
- **Imposible en Alemania, ~48% en Francia y España.** Ningún mecanismo de datos faltantes evita
  un país entero con esa precisión.
- **Efecto real y replicado** sobre el abandono.

**Tratamiento:** `balance` se conserva **sin imputar** y se añade `balance_zero` como bandera.
Rellenar con la media (119.827) el 36% de la base habría sido un desastre.

El nivel de saldo sí aporta algo, aunque poco: los quintiles recorren de 20,8% a 26,8% con forma
de U suave. Se conserva la variable continua y que decida el feature selection.

> **Advertencia para el notebook 04:** `balance_zero` está enredada con `geography` — si vale 1,
> el cliente no es alemán. No es colinealidad perfecta, pero hay que vigilarla.

### El efecto Alemania no se explica con estos datos

```
            edad    activos   productos    saldo*    abandono
France      38,5     51,7%      1,531      62.093     16,2%
Germany     39,8     49,7%      1,520     119.730     32,4%
Spain       38,9     53,0%      1,539      61.818     16,7%
```
<sub>*el saldo medio de Alemania solo parece distinto por ausencia de ceros: entre quienes tienen
saldo, los tres países promedian ~120.000.</sub>

Edad, actividad y productos son prácticamente idénticos. **Alemania es indistinguible de sus
vecinos en todo lo observable y aun así abandona al doble.** Comparando solo clientes con saldo,
sigue trece puntos por encima.

Dos lecturas, y el informe debe recoger ambas:

- **De negocio:** existe un factor del mercado alemán que el dataset no captura — competencia,
  regulación, producto. La recomendación sería recoger esos datos.
- **Metodológica:** que ninguna variable observable difiera y solo difiera el resultado es la
  huella típica de un generador sintético que asignó a Alemania otra probabilidad. Es la
  explicación más sencilla dado todo lo demás que hemos encontrado.

---

## D · Variables sensibles

El enunciado obliga a analizar `gender` y `geography` desde la perspectiva de sesgo y a justificar
su uso. Ya sabemos que geografía predice fuerte sin explicación. Falta el género — y de paso
`is_active_member`, que aún no habíamos medido.

In [0]:
print("— género —")
print(df.groupby("gender").agg(
    n=("exited", "size"),
    abandono=("exited", lambda s: round(s.mean()*100, 1)),
    edad_media=("age", "mean"),
    pct_activos=("is_active_member", "mean"),
    productos=("num_of_products", "mean"),
).round(3).to_string())

print("\n— género dentro de cada país —")
print(df.pivot_table(index="geography", columns="gender",
                     values="exited", aggfunc="mean").round(3).to_string())

print("\n— actividad —")
print(df.groupby("is_active_member")["exited"]
        .agg(abandono=lambda s: round(s.mean()*100, 1), n="size").to_string())

— género —
           n  abandono  edad_media  pct_activos  productos
gender                                                    
Female  4543      25.1      39.238        0.503      1.544
Male    5457      16.5      38.658        0.525      1.519

— género dentro de cada país —
gender     Female   Male
geography               
France      0.203  0.127
Germany     0.376  0.278
Spain       0.212  0.131

— actividad —
                  abandono     n
is_active_member                
0                     26.9  4849
1                     14.3  5151


### Género: misma estructura que Alemania

25,1% frente a 16,5%, más de diez errores estándar. Y otra vez edad, actividad y productos
prácticamente iguales: **ninguna variable observable explica la brecha**. Se replica en los tres
países (+7,6 / +9,8 / +8,1 pp), así que no es un artefacto de Alemania.

Dato de escala: los hombres alemanes (27,8%) abandonan más que las mujeres españolas (21,2%). El
efecto país pesa más que el efecto género.

### `is_active_member` es la variable más importante — y la única accionable

26,9% frente a 14,3%. Casi trece puntos, unos dieciséis errores estándar: el predictor binario más
fuerte del dataset.

Pero lo decisivo no es su fuerza. **La edad no se cambia, el país no se cambia, el género no se
cambia. La actividad sí.** Es la única variable sobre la que el banco puede intervenir, y por eso
sostiene todo el análisis prescriptivo del paso 7: la recomendación no será «vigilar a los mayores
de 50» —que no es una acción— sino «reactivar a los inactivos de 41 a 60 años».

> **Matiz obligado:** que la inactividad se asocie al abandono no prueba que reactivar lo evite.
> Podría ser síntoma y no causa — quien ya decidió irse deja de usar la cuenta. Distinguirlo
> requiere un experimento, no este dataset.

### Decisión 2 · Se entrenan dos versiones y se mide el coste

Género y geografía son fuertemente predictivos y ninguno tiene respaldo conductual. El modelo
aprendería «mujer alemana → riesgo alto» a partir de dos atributos que la persona no eligió.

Un matiz que invierte la intuición habitual: aquí la intervención es un **beneficio** — al cliente
de riesgo lo llaman y le ofrecen algo. Incluir el género hace que **más mujeres** entren en la
campaña; excluirlo haría que reciban menos ofertas pese a tener más riesgo real. El daño depende
del uso: si la misma tabla sirviera para *desinvertir* en quien se va, el signo se da la vuelta.

**Decisión:** el notebook 04 entrena el mismo pipeline dos veces, con y sin las variables
sensibles, y el 05 reporta la diferencia en recall. La conclusión toma la forma *«excluirlas
cuesta X puntos de recall, equivalente a N clientes en riesgo no detectados al mes»* — una cifra
que el negocio puede sopesar, en lugar de una decisión tomada por el analista.

---

## E · Las variables restantes

Queda `credit_score` sin mirar, y conviene medir `num_of_products`, que por el patrón de saldos se
intuye fuerte. También verificamos la predicción registrada en el notebook 01: dijimos que
`estimated_salary` tendría correlación ~0 **antes** de comprobarlo.

In [0]:
df["_cs"] = pd.qcut(df["credit_score"], 5, labels=["Q1","Q2","Q3","Q4","Q5"])
print("abandono por quintil de credit_score:")
print(df.groupby("_cs", observed=True)["exited"]
        .agg(abandono=lambda s: round(s.mean()*100, 1), n="size").to_string())
df.drop(columns="_cs", inplace=True)

print("\ncorrelaciones con exited (Spearman):")
for c in ["credit_score", "estimated_salary", "balance", "num_of_products", "age"]:
    print(f"  {c:<18} {df[c].corr(df['exited'], method='spearman'):+.4f}")

print("\nabandono por número de productos:")
print(df.groupby("num_of_products")["exited"]
        .agg(abandono=lambda s: round(s.mean()*100, 1), n="size").to_string())

abandono por quintil de credit_score:
     abandono     n
_cs                
Q1       22.5  2010
Q2       20.8  2020
Q3       19.7  2010
Q4       18.3  1981
Q5       20.5  1979

correlaciones con exited (Spearman):
  credit_score       -0.0233
  estimated_salary   +0.0121
  balance            +0.1111
  num_of_products    -0.1253
  age                +0.3240

abandono por número de productos:
                 abandono     n
num_of_products                
1                    27.7  5084
2                     7.6  4590
3                    82.7   266
4                   100.0    60


### El hallazgo metodológico del proyecto

```
1 producto  → 27,7%   (n=5084)
2 productos →  7,6%   (n=4590)
3 productos → 82,7%   (n=266)
4 productos → 100,0%  (n=60)
```

**Los sesenta clientes con cuatro productos abandonaron. Todos.** Y la correlación de Spearman de
esa variable es **−0,1253**: débil y de signo contrario.

Es la relación más determinante del dataset y la correlación dice que apenas existe. Una selección
de variables basada en correlación —que es lo que se hace por defecto— habría descartado el
predictor más decisivo.

Y no es un caso aislado. Las tres relaciones más fuertes son no monótonas:

| Variable | Spearman | Relación real |
|---|---|---|
| `age` | +0,324 | U invertida, pico del 56% en 51-60 |
| `num_of_products` | −0,125 | 7,6% con dos, 100% con cuatro |
| `balance` | +0,111 | U suave entre quintiles |

**Tesis del EDA:** en este dataset la selección de variables por correlación habría fallado. El
análisis por grupos es imprescindible, y los modelos de árbol parten con ventaja estructural sobre
los lineales.

### Separación perfecta: un problema técnico que hay que anticipar

`num_of_products = 4` con 100% de abandono produce **separación perfecta**. La estimación por
máxima verosimilitud de una regresión logística no converge en ese caso: el coeficiente óptimo
tiende a infinito.

`sklearn` no lanzará un error, porque aplica regularización L2 por defecto — pero producirá un
coeficiente desmesurado y un modelo excesivamente confiado sobre 60 observaciones.

Agrupar en `3+` lo resuelve: 326 clientes con 85,9% de abandono, extremo pero no perfecto.
`products_group` queda así justificada por **dos razones independientes**: tamaño muestral y
estabilidad numérica.

### Las demás

**`credit_score` es ruido.** Quintiles entre 18,3% y 22,5%, Spearman −0,023. Se conserva porque no
estorba, pero no se espera nada de él.

**`estimated_salary`: +0,0121 — predicción confirmada.** Quedó registrada en el notebook 01 antes
de medirla, a partir de su curtosis de −1,18 (distribución uniforme) y sus 9.999 valores únicos.

Eso liquida `balance_salary_ratio`: dividir una variable con señal entre ruido puro solo inyecta
ruido. Se descarta, igual que `tenure_age_ratio`.

---

## F · Transformaciones a Silver

### Qué se elimina

| Columna | Motivo |
|---|---|
| `row_number` | Identificador técnico, cardinalidad 100% |
| `surname` | Prohibido por el enunciado; cardinalidad 29,3%; sin valor predictivo |
| metadatos `_*` | Trazabilidad de ingesta; se conservan en Bronze |

`customer_id` **sí se mantiene**: es necesario para devolver las predicciones al sistema
operacional en el notebook 06.

### Qué se crea

| Feature | Definición | Justificación |
|---|---|---|
| `balance_zero` | `balance = 0` | Estado distinto, no nulo. Hueco en la distribución e imposibilidad en Alemania |
| `age_group` | <30 / 30-40 / 40-50 / 50-60 / 60+ | Los modelos lineales no pueden representar la U invertida sin ella |
| `products_group` | 1 / 2 / 3+ | n insuficiente en 3 y 4; evita la separación perfecta |
| `credit_score_band` | tramos crediticios estándar | Traduce el score a lenguaje de negocio para el dashboard |

### Qué se descartó y por qué

| Feature | Motivo |
|---|---|
| `tenure_age_ratio` | Sus dos componentes son independientes (ρ = −0,01): el concepto no existe |
| `balance_salary_ratio` | `estimated_salary` es ruido uniforme (ρ = +0,012) |
| bandera de incoherencia | Confusor demostrado: 7,0% vs 7,5% dentro del mismo tramo de edad |

### Lo que NO se hace aquí

Ni imputación, ni codificación, ni escalado, ni balanceo. Todo eso **aprende estadísticos de los
datos** y debe ajustarse únicamente sobre el conjunto de entrenamiento, dentro del `Pipeline` de
sklearn del notebook 04. Meterlo en Silver sería fuga de datos.

In [0]:
from pyspark.sql import functions as F

silver = (
    bronze
    # --- eliminaciones ---------------------------------------------------
    .drop("row_number", "surname", "_ingested_at", "_source_file",
          "_source_url", "_extracted_at")

    # --- tipado: enteros 0/1 a booleano ----------------------------------
    .withColumn("has_cr_card",      F.col("has_cr_card")      == 1)
    .withColumn("is_active_member", F.col("is_active_member") == 1)
    .withColumn("exited",           F.col("exited")           == 1)
    .withColumn("balance",          F.col("balance").cast("double"))
    .withColumn("estimated_salary", F.col("estimated_salary").cast("double"))

    # --- features deterministas ----------------------------------------
    .withColumn("balance_zero", F.col("balance") == 0)
    .withColumn("age_group",
        F.when(F.col("age") < 30, "18-29")
         .when(F.col("age") < 40, "30-39")
         .when(F.col("age") < 50, "40-49")
         .when(F.col("age") < 60, "50-59")
         .otherwise("60+"))
    .withColumn("products_group",
        F.when(F.col("num_of_products") == 1, "1")
         .when(F.col("num_of_products") == 2, "2")
         .otherwise("3+"))
    .withColumn("credit_score_band",
        F.when(F.col("credit_score") < 580, "poor")
         .when(F.col("credit_score") < 670, "fair")
         .when(F.col("credit_score") < 740, "good")
         .when(F.col("credit_score") < 800, "very_good")
         .otherwise("excellent"))

    .withColumn("_silver_at", F.current_timestamp())
)

silver.printSchema()
print("filas:", silver.count())

root
 |-- customer_id: long (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- geography: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- balance: double (nullable = true)
 |-- num_of_products: integer (nullable = true)
 |-- has_cr_card: boolean (nullable = true)
 |-- is_active_member: boolean (nullable = true)
 |-- estimated_salary: double (nullable = true)
 |-- exited: boolean (nullable = true)
 |-- balance_zero: boolean (nullable = true)
 |-- age_group: string (nullable = false)
 |-- products_group: string (nullable = false)
 |-- credit_score_band: string (nullable = false)
 |-- _silver_at: timestamp (nullable = false)

filas: 10000


In [0]:
# Validaciones antes de escribir. Si alguna falla, no se persiste nada.
s = silver

assert s.count() == 10000,                       "se perdieron filas"
assert s.select("customer_id").distinct().count() == 10000, "customer_id dejó de ser único"
assert "surname" not in s.columns,               "surname se propago a Silver"
assert "row_number" not in s.columns,            "row_number se propago a Silver"

nulos = s.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in s.columns]) \
         .toPandas().T.rename(columns={0: "n_null"})
assert nulos["n_null"].sum() == 0,               f"aparecieron nulos:\n{nulos[nulos.n_null > 0]}"

# Las features derivadas deben cubrir el 100% de las filas
for col, esperado in [("age_group", 5), ("products_group", 3), ("credit_score_band", 5)]:
    n = s.select(col).distinct().count()
    print(f"  {col:<20} {n} categorías (esperadas <= {esperado})")

print("\nvalidaciones superadas")

  age_group            5 categorias (esperadas <= 5)
  products_group       3 categorias (esperadas <= 3)
  credit_score_band    5 categorias (esperadas <= 5)

validaciones superadas


In [0]:
(
    silver.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

spark.sql(f"""
    COMMENT ON TABLE {SILVER_TABLE} IS
    'Silver. Datos tipados y limpios desde Bronze, con features deterministicas.
     Sin imputación, codificación ni escalado: todo lo que aprende estadísticos vive
     en el Pipeline de sklearn del notebook 04 para evitar fuga de datos.'
""")

print("Silver escrita:", SILVER_TABLE)
display(spark.table(SILVER_TABLE).limit(5))

Silver escrita: bank_churn.silver.bank_customers_clean


customer_id,credit_score,geography,gender,age,tenure,balance,num_of_products,has_cr_card,is_active_member,estimated_salary,exited,balance_zero,age_group,products_group,credit_score_band,_silver_at
15634602,619,France,Female,42,2,0.0,1,true,true,101348.88,true,true,40-49,1,fair,2026-08-02T02:53:59.361Z
15647311,608,Spain,Female,41,1,83807.86,1,false,true,112542.58,false,false,40-49,1,fair,2026-08-02T02:53:59.361Z
15619304,502,France,Female,42,8,159660.8,3,true,false,113931.57,true,false,40-49,3+,poor,2026-08-02T02:53:59.361Z
15701354,699,France,Female,39,1,0.0,2,false,false,93826.63,false,true,30-39,2,good,2026-08-02T02:53:59.361Z
15737888,850,Spain,Female,43,2,125510.82,1,true,true,79084.1,false,false,40-49,1,excellent,2026-08-02T02:53:59.361Z


## G · Perfil de calidad — estado «después»

La rúbrica pide comparar contra el «antes» de la sección A.

In [0]:
sdf = spark.table(SILVER_TABLE).toPandas()
cols = [c for c in sdf.columns if not c.startswith("_")]

profile_after = pd.DataFrame({
    "dtype":    sdf[cols].dtypes.astype(str),
    "n_null":   sdf[cols].isna().sum(),
    "n_unique": sdf[cols].nunique(),
})

print("ANTES :", len(profile_before), "columnas")
print("DESPUÉS:", len(profile_after), "columnas")
print("\neliminadas:", sorted(set(profile_before.index) - set(profile_after.index)))
print("creadas   :", sorted(set(profile_after.index) - set(profile_before.index)))
profile_after

ANTES : 14 columnas
DESPUES: 16 columnas

eliminadas: ['row_number', 'surname']
creadas   : ['age_group', 'balance_zero', 'credit_score_band', 'products_group']


,dtype,n_null,n_unique
customer_id,int64,0,10000
credit_score,int32,0,460
geography,object,0,3
gender,object,0,2
age,int32,0,70
tenure,int32,0,11
balance,float64,0,6382
num_of_products,int32,0,4
has_cr_card,bool,0,2
is_active_member,bool,0,2


In [0]:
# Distribución de las features nuevas: ninguna categoría debe quedar despoblada
for col in ["age_group", "products_group", "credit_score_band", "balance_zero"]:
    t = (sdf.groupby(col)
            .agg(n=("exited", "size"), abandono=("exited", lambda s: round(s.mean()*100, 1)))
            .sort_values("n", ascending=False))
    print(f"— {col} —")
    print(t.to_string(), "\n")

— age_group —
              n  abandono
age_group                
30-39      4346      10.9
40-49      2618      30.8
18-29      1641       7.6
50-59       869      56.0
60+         526      27.9 

— products_group —
                   n  abandono
products_group                
1               5084      27.7
2               4590       7.6
3+               326      85.9 

— credit_score_band —
                      n  abandono
credit_score_band                
fair               3331      20.6
good               2428      18.6
poor               2362      22.0
very_good          1224      20.6
excellent           655      19.5 

— balance_zero —
                 n  abandono
balance_zero                
False         6383      24.1
True          3617      13.8 



## Resultado

```
bronze.bank_customers_raw  ──▶  silver.bank_customers_clean
      18 columnas                    16 columnas
      copia intocable                tipada, limpia, con 4 features derivadas
```

**Decisiones tomadas, todas con evidencia:**

1. `balance = 0` es un estado legítimo → sin imputar, más bandera `balance_zero`
2. Las variables sensibles se evalúan entrenando dos versiones y midiendo el coste en recall
3. `products_group` por tamaño muestral y por separación perfecta
4. `age_group` porque los modelos lineales no pueden representar la U invertida
5. `tenure_age_ratio` y `balance_salary_ratio` descartadas: sus componentes son ruido

**Para el informe** — de este notebook salen: el perfil antes/después, el aislamiento del confusor
de país, la tabla de no monotonía y la justificación de cada feature.

**Siguiente:** `03_eda_analisis` — formalizar estos hallazgos con visualizaciones e inferencia.
Dos pruebas ya identificadas: chi-cuadrado de `tenure` contra `exited` (que **no** debería
rechazar) y contraste del efecto de `is_active_member`.